# Model A: EfficientNetB0 baseline without augmentation

This notebook establishes the clean-image baseline for the project. I train only the classification head of an ImageNet-pretrained EfficientNetB0 and do not apply random image augmentation.

The original dataset already has `train` and `test` folders. I split the `train` folder into training and validation subsets, while keeping the `test` folder untouched until the final evaluation. This prevents the test set from influencing early stopping or checkpoint selection.

In [ ]:
from pathlib import Path
import json
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

SEED = 42
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    # Some TensorFlow versions do not expose this option.
    pass

print(f"Python: {sys.version.split()[0]}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")

## Paths and experiment settings

The dataset is expected to have the structure below. Change `DRIVE_DATASET_DIR` if your folder is stored elsewhere.

```text
Emotions Dataset/
├── train/
│   ├── angry/
│   ├── happy/
│   └── sad/
└── test/
    ├── angry/
    ├── happy/
    └── sad/
```

In [ ]:
CLASS_NAMES = ['angry', 'happy', 'sad']
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/Emotions Dataset')
LOCAL_DATASET_DIR = Path('/content/emotions_dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/CNN-Robustness-Low-Light-Analysis/outputs/model_a')

CONFIG = {
    'image_size': (224, 224),
    'batch_size': 32,
    'validation_split': 0.20,
    'epochs': 30,
    'learning_rate': 1e-3,
    'dropout_rate': 0.30,
    'dense_units': 256,
}

CONFIG

## Copy the dataset to the Colab runtime

Reading thousands of small files directly from Google Drive is slow. I copy the dataset once to the temporary Colab disk, but save models and results back to Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

required_directories = [
    DRIVE_DATASET_DIR / split / class_name
    for split in ['train', 'test']
    for class_name in CLASS_NAMES
]
missing_directories = [path for path in required_directories if not path.is_dir()]

if missing_directories:
    missing_text = '\n'.join(str(path) for path in missing_directories)
    raise FileNotFoundError(f"The following dataset folders were not found:\n{missing_text}")

if not LOCAL_DATASET_DIR.exists():
    print('Copying the dataset from Drive to the Colab runtime...')
    shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
    print('Copy complete.')
else:
    print('Using the dataset already copied to the Colab runtime.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_DIR = LOCAL_DATASET_DIR / 'train'
TEST_DIR = LOCAL_DATASET_DIR / 'test'

## Build the data pipeline

Only resizing and batching are performed here. EfficientNetB0 already contains its own input rescaling layer, so the datasets keep their original image range of approximately 0–255.

In [ ]:
common_dataset_options = {
    'image_size': CONFIG['image_size'],
    'batch_size': CONFIG['batch_size'],
    'label_mode': 'categorical',
    'class_names': CLASS_NAMES,
}

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=CONFIG['validation_split'],
    subset='training',
    seed=SEED,
    shuffle=True,
    **common_dataset_options,
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=CONFIG['validation_split'],
    subset='validation',
    seed=SEED,
    shuffle=True,
    **common_dataset_options,
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    **common_dataset_options,
)

test_file_paths = list(test_dataset.file_paths)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [ ]:
images, labels = next(iter(validation_dataset))
print(f"Batch shape: {images.shape}")
print(f"Pixel range: {images.numpy().min():.1f} to {images.numpy().max():.1f}")

plt.figure(figsize=(12, 6))
for index in range(min(8, len(images))):
    plt.subplot(2, 4, index + 1)
    plt.imshow(images[index].numpy().astype('uint8'))
    label_index = int(tf.argmax(labels[index]))
    plt.title(CLASS_NAMES[label_index])
    plt.axis('off')
plt.suptitle('Validation examples (no augmentation)')
plt.tight_layout()
plt.show()

## Define Model A

For this baseline, I freeze the ImageNet-pretrained EfficientNetB0 backbone and train a small classification head. I call the backbone with `training=False` so its Batch Normalization statistics remain fixed during this stage.

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=CONFIG['image_size'] + (3,),
)
base_model.trainable = False

inputs = tf.keras.Input(shape=CONFIG['image_size'] + (3,), name='image')
features = base_model(inputs, training=False)
features = layers.GlobalAveragePooling2D(name='global_average_pooling')(features)
features = layers.Dropout(CONFIG['dropout_rate'], name='dropout')(features)
features = layers.Dense(CONFIG['dense_units'], activation='relu', name='classifier_dense')(features)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax', name='emotion')(features)

model = tf.keras.Model(inputs, outputs, name='model_a_baseline')
model.summary()

trainable_parameters = sum(np.prod(variable.shape) for variable in model.trainable_weights)
print(f"Trainable parameters: {trainable_parameters:,}")

## Train the classification head

The best checkpoint is selected using validation loss. The test set is not passed to `fit()` and does not influence training.

In [ ]:
checkpoint_path = OUTPUT_DIR / 'model_a_baseline.keras'
training_log_path = OUTPUT_DIR / 'training_log.csv'

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top_2_accuracy'),
    ],
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        mode='min',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(training_log_path)),
]

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=CONFIG['epochs'],
    callbacks=callbacks,
    verbose=2,
)

## Final evaluation on the untouched clean test set

I reload the checkpoint selected by validation loss before evaluating. This makes the reported test metrics correspond to the model that is actually saved.

In [ ]:
best_model = tf.keras.models.load_model(str(checkpoint_path))
test_metrics = best_model.evaluate(test_dataset, return_dict=True, verbose=1)

print('\nClean test metrics')
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
probabilities = best_model.predict(test_dataset, verbose=1)
predicted_labels = np.argmax(probabilities, axis=1)
true_labels = np.concatenate([
    np.argmax(batch_labels.numpy(), axis=1)
    for _, batch_labels in test_dataset
])

report = classification_report(
    true_labels,
    predicted_labels,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
    output_dict=True,
)

print(classification_report(
    true_labels,
    predicted_labels,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

predictions = pd.DataFrame({
    'file': test_file_paths,
    'true_label': [CLASS_NAMES[index] for index in true_labels],
    'predicted_label': [CLASS_NAMES[index] for index in predicted_labels],
})
for class_index, class_name in enumerate(CLASS_NAMES):
    predictions[f'prob_{class_name}'] = probabilities[:, class_index]

predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

## Inspect and save the results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs_ran = range(1, len(history.history['loss']) + 1)

axes[0].plot(epochs_ran, history.history['accuracy'], label='Training')
axes[0].plot(epochs_ran, history.history['val_accuracy'], label='Validation')
axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(epochs_ran, history.history['loss'], label='Training')
axes[1].plot(epochs_ran, history.history['val_loss'], label='Validation')
axes[1].set(title='Cross-entropy loss', xlabel='Epoch', ylabel='Loss')
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.suptitle('Model A training history')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

matrix = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title('Model A: clean test confusion matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
history_to_save = {
    name: [float(value) for value in values]
    for name, values in history.history.items()
}

with open(OUTPUT_DIR / 'training_history.json', 'w') as file:
    json.dump(history_to_save, file, indent=2)

best_epoch = int(np.argmin(history.history['val_loss']) + 1)
results_summary = {
    'experiment': 'Model A - baseline without augmentation',
    'seed': SEED,
    'tensorflow_version': tf.__version__,
    'image_size': list(CONFIG['image_size']),
    'batch_size': CONFIG['batch_size'],
    'validation_split': CONFIG['validation_split'],
    'epochs_completed': len(history.history['loss']),
    'best_epoch_by_validation_loss': best_epoch,
    'best_validation_loss': float(min(history.history['val_loss'])),
    'best_validation_accuracy': float(max(history.history['val_accuracy'])),
    'test_loss': float(test_metrics['loss']),
    'test_accuracy': float(test_metrics['accuracy']),
    'test_top_2_accuracy': float(test_metrics['top_2_accuracy']),
    'test_macro_f1': float(report['macro avg']['f1-score']),
    'test_weighted_f1': float(report['weighted avg']['f1-score']),
    'classification_report': report,
}

with open(OUTPUT_DIR / 'results_summary.json', 'w') as file:
    json.dump(results_summary, file, indent=2)

print(json.dumps({
    'best_epoch': results_summary['best_epoch_by_validation_loss'],
    'test_accuracy': results_summary['test_accuracy'],
    'test_macro_f1': results_summary['test_macro_f1'],
    'test_top_2_accuracy': results_summary['test_top_2_accuracy'],
}, indent=2))
print(f"\nSaved all Model A outputs to: {OUTPUT_DIR}")

## What to keep from this run

For the next experiment, keep the complete `outputs/model_a` folder. In particular, the next notebooks will need `model_a_baseline.keras`, `results_summary.json`, `training_history.json`, `training_curves.png`, and `confusion_matrix.png`.

When reporting this baseline, use the metrics from `results_summary.json`, not the last line of the training history.